# Init & functions

In [13]:
from src.prompt_manager import PromptManager, PromptSuite, PromptTemplate, create_empty_prompt_template, create_chat_prompt_dict
from pathlib import Path

In [14]:

def build_formative_suite(
    config: dict,
    suite_description: str,
    scale_size: int,
    system_prompt_template: str
) -> PromptSuite:
    """
    Parses a configuration dictionary to build a PromptSuite for psychometric evaluation.
    Utilizes .format() to dynamically inject variables into interchangeable system prompt templates.
    """
    prompt_templates = []

    # Dynamically generate the scale tag (e.g., "scale_7")
    dynamic_scale_tag = f"scale_{scale_size}"

    for field_key, field_config in config.items():
        human_name = field_config["human_name"]
        user_template = field_config["user_template"]
        base_tags = field_config["tags"]

        for dim_key, dim_data in field_config["dimensions"].items():

            # Clean the dimension key for the prompt (e.g., "creator_brand_alignment" -> "creator brand alignment"), and capitalize
            dim_key_clean = dim_key.replace('_', ' ').title()

            # 1. Construct the System Prompt via .format()
            system_prompt = system_prompt_template.format(
                human_name=human_name,
                dim_key_clean=dim_key_clean,
                definition=dim_data['definition'],
                rubric=dim_data['rubric']
            )

            # 2. Assemble the Template Dictionary
            templ_dict = create_chat_prompt_dict(
                name=f"rubric_{field_key}_{dim_key}",
                description=f"Evaluates the intensity of {dim_key} on the {human_name}",
                template_chat={
                    "system": system_prompt.strip(),
                    "user": user_template.strip()
                },
                # Create a new list object in memory for each iteration to avoid YAML aliasing
                token_constraints=[str(i) for i in range(1, scale_size + 1)],
                template_tags=['rubric', dynamic_scale_tag,
                               'trait_intensity', dim_key] + base_tags,
                dimension_name=f"{field_key}_{dim_key}"
            )

            # 3. Append to list
            prompt_templates.append(PromptTemplate.from_dict(templ_dict))

    # Return the fully compiled suite
    metadata = {'description': suite_description}
    return PromptSuite.from_list(prompt_templates, metadata=metadata)

# __Barter deals__

## Comments

- We use scale size 7, as it increases the 'resolution' of the evaluations (semantic zooming). Need to check whether it indeed improved the model fit though.

## Init

In [15]:
suite_folder = Path("../prompts/PromptSuites/BARTER_DEALS")
pm = PromptManager(suite_folder)

PromptManager initialized with folder: ..\prompts\PromptSuites\BARTER_DEALS


In [16]:
SYSTEM_PROMPT_BARTER = """You are an expert evaluator of influencer marketing deals ("Deal Texts") for Barter, an app-based creator marketplace. 
In this ecosystem, creators scroll a feed of deals and apply to collaborate. If accepted, they produce content (e.g., UGC, TikToks, Instagram Reels) primarily in exchange for the brand's offering, with occasional cash supplements. Your evaluation must be grounded in this creator ecosystem.

You will be provided with a Deal Text consisting of a Title, Body, and Requirements. Please analyze this text, focusing strictly on the following dimension: **{dim_key_clean}**.

Dimension Definition:
{definition}

Scale Definition:
{rubric}

Evaluate where the text falls on this continuum, and provide the closest whole number rating (1-7). """

In [17]:
SYSTEM_PROMPT_BARTER_NEW2 = """You are an expert evaluator of influencer marketing deals ("Deal Texts") for Barter, an app-based creator marketplace.
In this ecosystem, creators scroll a feed of deals and apply to collaborate. If accepted, they produce content (e.g., UGC, TikToks, Instagram Reels) primarily in exchange for the brand's offering, with occasional cash supplements. Ground your evaluation in this creator marketplace context. Use both the information in the Deal Text and your general world knowledge when needed.

You will be given one Deal Text with a Title, Body, and Requirements. Evaluate it on exactly one dimension: {dim_key_clean}.

Dimension Definition:
{definition}

Scale Definition:
{rubric}

Evaluate where the deal falls on this continuum and return the closest whole-number rating from 1 to 7."""

## Holistic

### Naive baseline

In [18]:
EVALUATION_CONFIG_NAIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Title: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "naive_holistic"],
        "dimensions": {
            "overall_deal_attractiveness": {
                "definition": "This evaluates the overall appeal of the barter opportunity from the perspective of a typical content creator. It assesses the total subjective value of the offer, ranging from a fundamentally undesirable deal that offers no compelling reward, up to a highly coveted, premium opportunity that is instantly desirable.",
                "rubric": "1: Extremely Weak (A fundamentally undesirable deal offering no compelling value or appeal)\n2: Weak\n3: Slightly Weak\n4: Fair / Average (A standard, typical deal with baseline market appeal)\n5: Slightly Strong\n6: Strong\n7: Excellent (A highly coveted, premium opportunity that is instantly desirable)"
            }
        }
    }
}
# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_NAIVE,
    suite_description="Holistic Naive Quality, scale 7, SPV3",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_BARTER)
holistic_suite.save(suite_folder, filename="holistic_naive_quality_S7_SPV3")

✅ Saved suite to ..\prompts\PromptSuites\BARTER_DEALS\holistic_naive_quality_S7_SPV3_suite_dbadc59142e9.yml


WindowsPath('../prompts/PromptSuites/BARTER_DEALS/holistic_naive_quality_S7_SPV3_suite_dbadc59142e9.yml')

### Informed baseline

In [19]:
EVALUATION_CONFIG_MACRO_FORMATIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Title: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "informed_holistic"],
        "dimensions": {
            "overall_reward_value": {
                "definition": "Overall Reward Value is defined as the total desirability and worth of the compensation package being offered to the creator. This measures the continuum between a highly unappealing reward with little to no worth, versus an exceptionally premium, highly coveted offering.",
                "rubric": "1: Highly Unappealing (A very poor offering with negligible worth or desirability)\n2: Very Low Appeal\n3: Low Appeal\n4: Standard Market Value (An average, typical compensation package)\n5: High Appeal\n6: Very High Appeal\n7: Highly Premium Offering (An exceptionally valuable and highly coveted reward)"
            },
            "overall_task_demand": {
                "definition": "Overall Task Demand is defined as the total amount of effort, time, and rigid compliance required to complete the campaign. This measures the continuum between a very fast, easy, and flexible request, versus a highly demanding, time-consuming project with strict rules.",
                "rubric": "1: Minimal Demand (An extremely fast, easy task with total flexibility)\n2: Very Low Demand\n3: Low Demand\n4: Moderate Demand (A standard campaign requiring an average amount of time and effort)\n5: High Demand\n6: Very High Demand\n7: Extreme Demand (A highly time-consuming project with intensive requirements and strict rules)"
            },
            "overall_pitch_quality": {
                "definition": "Overall Pitch Quality is defined as the clarity, structure, and professional tone of the brand's written communication. This measures the continuum between a highly confusing, poorly written, or unprofessional text, versus a flawlessly structured, perfectly clear, and highly professional proposal.",
                "rubric": "1: Extremely Poor Quality (The text is highly confusing, unstructured, or deeply unprofessional)\n2: Very Low Quality\n3: Low Quality\n4: Standard Quality (A moderately clear, average business communication)\n5: High Quality\n6: Very High Quality\n7: Exceptional Quality (A perfectly clear, flawlessly structured, and highly professional proposal)"
            }
        }
    }
}


diagnostic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_MACRO_FORMATIVE,
    suite_description="Informed holistic NoBARS, revised dimensions, including requirements,scale 7, revision3, system prompt V3",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_BARTER)
diagnostic_suite.save(
    suite_folder, filename="holistic_informed_v2_continuous_noBARS_wrequirements_scale7_SPV3")

✅ Saved suite to ..\prompts\PromptSuites\BARTER_DEALS\holistic_informed_v2_continuous_noBARS_wrequirements_scale7_SPV3_suite_38c4714f0e3a.yml


WindowsPath('../prompts/PromptSuites/BARTER_DEALS/holistic_informed_v2_continuous_noBARS_wrequirements_scale7_SPV3_suite_38c4714f0e3a.yml')

## Formative

In [20]:
EVALUATION_CONFIG_FORMATIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Here is the Deal Text:\n\nTitle: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "formative"],
        "dimensions": {
            "brand_prestige": {
                "definition": "Brand Prestige is defined as how well-known and trusted the brand name is in the market. This measures the continuum between a brand that is completely unknown or suspicious, versus a world-famous brand that is a clear global leader in its industry.",
                "rubric": "1: Unknown or Suspicious (No one recognizes this brand name; it feels untrustworthy)\n2: Very Low Recognition\n3: Low Recognition\n4: Average/Niche Recognition (Known only in a very small or specific area)\n5: High Recognition\n6: Very High Recognition\n7: World-Famous/Global Leader (The brand name is recognized by almost everyone)"
            },
            "perceived_economic_value": {
                "definition": "Perceived Economic Value is defined as the estimated retail value and premium nature of the product, service, or compensation offered to the creator. This measures the continuum between a reward of negligible, near-zero financial worth, versus a highly expensive, premium, or luxury offering.",
                "rubric": "1: Negligible Value (A very cheap or nearly worthless offering)\n2: Very Low Value\n3: Low Value\n4: Moderate Value (A standard, average-priced product or service)\n5: High Value\n6: Very High Value\n7: Exceptional/Luxury Value (A highly expensive or premium reward)"
            },
            "creator_brand_alignment": {
                "definition": "Creator Brand Alignment is defined as how naturally the product blends into a professional content creator's social media feed. This measures the continuum between a product that visually or conceptually clashes and looks completely out of place, versus a product that feels like a seamless, natural addition to the creator's curated image.",
                "rubric": "1: Complete Clash (The product looks completely out of place and visually or conceptually jarring)\n2: Poor Fit\n3: Slightly Awkward Fit\n4: Neutral Fit (A standard, acceptable product that neither clashes nor perfectly blends)\n5: Good Fit\n6: High Harmony\n7: Perfect Harmony (A seamless, completely natural addition to a curated image)"
            },
            "functional_utility": {
                "definition": "Functional Utility is defined as the practical, everyday usefulness and utilitarian value of the physical product or service being offered. This measures the continuum between a purely decorative, novelty, or hedonic item versus a highly functional necessity that solves a practical daily problem.",
                "rubric": "1: Purely Novelty/Decorative (Zero practical utility or everyday use)\n2: Very Low Utility\n3: Low Utility\n4: Moderate Utility (A standard item with some practical, but not strictly essential, daily use)\n5: High Utility\n6: Very High Utility\n7: Essential Necessity (A highly practical item that solves a clear, unavoidable daily problem)"
            },
            "product_universality": {
                "definition": "Product Universality is defined as the mass-market consumer appeal of the physical product or service on offer. This measures the continuum between an item designed for a highly restricted, specialized target audience versus an everyday product with universal utility across a broad consumer base.",
                "rubric": "1: Extremely Niche (Highly restricted to a very specialized or rare target audience)\n2: Very Niche\n3: Somewhat Niche\n4: Moderate Appeal (A standard product with average, everyday consumer appeal)\n5: Broad Appeal\n6: Very Broad Appeal\n7: Universal Mass-Market Appeal (An everyday item with near-universal relevance to the general public)"
            },
            "workload_volume": {
                "definition": "Workload Volume is defined as the total time and effort required to execute the collaboration. This measures the continuum between a rapid, low-effort task requiring minimal time and energy, versus a demanding commitment requiring extensive time and high intensity effort.",
                "rubric": "1: Minimal Effort (Negligible time and energy required)\n2: Very Light Effort\n3: Light Effort\n4: Moderate Effort (A standard, manageable investment of time and energy)\n5: Heavy Effort\n6: Very Heavy Effort\n7: Maximum Effort (Extensive time and high intensity energy investment)"
            },
            "creative_restrictiveness": {
                "definition": "Creative Restrictiveness is defined as the level of brand control over the creative process. This measures the continuum between total freedom for the creator to choose their own style and ideas, versus strict rules where the brand decides exactly how the content must look and sound.",
                "rubric": "1: Total Freedom (The creator has full control over the style and ideas)\n2: Very High Freedom\n3: High Freedom\n4: Moderate Rules (A standard balance of freedom and brand guidelines)\n5: Strict Rules\n6: Very Strict Rules\n7: Total Brand Control (The brand decides every detail of the content)"
            },
            "call_to_action_strength": {
                "definition": "Call-to-Action Strength is defined as the level of excitement and warmth in the brand's invitation to collaborate. This measures the continuum between a cold, distant, or purely business-like closing, versus a very welcoming and enthusiastic invitation to work together.",
                "rubric": "1: Extremely Cold and Distant (Purely business-like; no warmth)\n2: Very Weak Excitement\n3: Weak Excitement\n4: Neutral/Average Invitation (A standard, professional closing)\n5: Strong Excitement\n6: Very Strong Excitement\n7: Maximum Excitement and Warmth (A very welcoming and enthusiastic invitation)"
            },
            "rhetorical_objectivity": {
                "definition": "Rhetorical Objectivity is defined as the level of factual realism versus promotional marketing hype in the pitch. This measures the continuum between a text that relies entirely on exaggerated claims, heavy enthusiasm, and sales hype, versus a text that is completely literal, neutral, and strictly focused on business facts.",
                "rubric": "1: Pure Promotional Hyperbole (Entirely driven by sales hype, exaggerated claims, and heavy enthusiasm)\n2: Highly Promotional\n3: Moderately Promotional\n4: Neutral/Standard Pitch (A standard balance of normal marketing warmth and clear factual details)\n5: Grounded and Pragmatic\n6: Highly Objective and Realistic\n7: Purely Factual and Literal (Strictly focused on business facts and constraints with zero promotional hype)"
            },
            "proposition_clarity": {
                "definition": "Proposition Clarity is defined as the structural clarity and cognitive ease of processing the pitch. This measures the continuum between a disorganized, vague, or confusing text requiring high cognitive effort to decipher the terms of trade, versus a highly structured, explicit pitch where the required deliverables and offered rewards are immediately obvious.",
                "rubric": "1: Extremely Confusing and Opaque (Highly disorganized; the required deliverables and rewards are very difficult to decipher)\n2: Very Unclear\n3: Slightly Unclear\n4: Average Clarity (A standard, readable pitch where the main terms are understandable with normal cognitive effort)\n5: Clear and Structured\n6: Very Clear\n7: Perfectly Explicit and Immediately Obvious (Highly structured; the exact deliverables and rewards are instantly clear with zero cognitive effort)"
            }
        }
    }
}

# New system prompt
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_FORMATIVE,
    suite_description="Formative NoBARS, revised dimensions, including requirements,scale 7, revision2, system prompt V3, user prompt V2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_BARTER)
formative_suite.save(
    suite_folder, filename="formative_v2_continuous_noBARS_wrequirements_scale7_SPV3_UPV2")

✅ Saved suite to ..\prompts\PromptSuites\BARTER_DEALS\formative_v2_continuous_noBARS_wrequirements_scale7_SPV3_UPV2_suite_1e1ff82a28e2.yml


WindowsPath('../prompts/PromptSuites/BARTER_DEALS/formative_v2_continuous_noBARS_wrequirements_scale7_SPV3_UPV2_suite_1e1ff82a28e2.yml')

Formative new, FINAL REWRITE PLSSSS

In [21]:
EVALUATION_CONFIG_FORMATIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Here is the Deal Text:\n\nTitle: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "formative"],
        "dimensions": {
            "brand_prestige": {
                "definition": "Brand Prestige is defined as the degree to which the brand is recognized, established, and trusted in the market. This ranges from a brand with very low recognition or credibility to a brand with very high recognition, reputation, and perceived status.",
                "rubric": "1: Very low prestige\n2: Low prestige\n3: Slightly below-average prestige\n4: Moderate prestige\n5: Above-average prestige\n6: High prestige\n7: Very high prestige"
            },
            "perceived_economic_value": {
                "definition": "Perceived Economic Value is defined as the perceived financial value, premium level, or worth of the product, service, or compensation offered to the creator. This ranges from very low value to very high value.",
                "rubric": "1: Very low value\n2: Low value\n3: Slightly below-average value\n4: Moderate value\n5: Above-average value\n6: High value\n7: Very high value"
            },
            "creator_content_fit": {
                "definition": "Creator Content Fit is defined as how naturally the collaboration opportunity fits the kind of content creators would realistically make and share. This ranges from a deal that feels very difficult or unnatural to integrate into creator content to a deal that feels very natural and easy to integrate into creator content.",
                "rubric": "1: Very poor fit\n2: Poor fit\n3: Slightly below-average fit\n4: Moderate fit\n5: Above-average fit\n6: Good fit\n7: Excellent fit"
            },
            "functional_utility": {
                "definition": "Functional Utility is defined as the degree to which the product or service offers practical, everyday usefulness. This ranges from very low practical usefulness to very high practical usefulness.",
                "rubric": "1: Very low utility\n2: Low utility\n3: Slightly below-average utility\n4: Moderate utility\n5: Above-average utility\n6: High utility\n7: Very high utility"
            },
            "product_universality": {
                "definition": "Product Universality is defined as the breadth of the product's or service's general consumer appeal. This ranges from very narrow appeal to very broad appeal.",
                "rubric": "1: Very narrow appeal\n2: Narrow appeal\n3: Slightly below-average appeal breadth\n4: Moderate appeal breadth\n5: Broad appeal\n6: Very broad appeal\n7: Near-universal appeal"
            },
            "workload_volume": {
                "definition": "Workload Volume is defined as the total amount of time, effort, and work required to complete the collaboration. This ranges from very low workload to very high workload.",
                "rubric": "1: Very low workload\n2: Low workload\n3: Slightly below-average workload\n4: Moderate workload\n5: Above-average workload\n6: High workload\n7: Very high workload"
            },
            "creative_restrictiveness": {
                "definition": "Creative Restrictiveness is defined as the degree to which the collaboration limits the creator's creative freedom. This ranges from very high freedom and very low restriction to very low freedom and very high restriction.",
                "rubric": "1: Very low restrictiveness\n2: Low restrictiveness\n3: Slightly below-average restrictiveness\n4: Moderate restrictiveness\n5: Above-average restrictiveness\n6: High restrictiveness\n7: Very high restrictiveness"
            },
            "call_to_action_strength": {
                "definition": "Call-to-Action Strength is defined as the degree of warmth, enthusiasm, and inviting energy in the brand's invitation to collaborate. This ranges from very weak or distant to very strong and enthusiastic.",
                "rubric": "1: Very weak call to action\n2: Weak call to action\n3: Slightly below-average call to action strength\n4: Moderate call to action strength\n5: Above-average call to action strength\n6: Strong call to action\n7: Very strong call to action"
            },
            "promotional_hype_level": {
                "definition": "Promotional Hype Level is defined as the degree to which the pitch uses promotional, exaggerated, or sales-oriented language rather than a neutral, factual, and business-like tone. This ranges from very low hype to very high hype.",
                "rubric": "1: Very low hype\n2: Low hype\n3: Slightly below-average hype\n4: Moderate hype\n5: Above-average hype\n6: High hype\n7: Very high hype"
            },
            "proposition_clarity": {
                "definition": "Proposition Clarity is defined as how easy the pitch is to understand in terms of what is being offered and what is being requested. This ranges from very unclear to very clear.",
                "rubric": "1: Very unclear\n2: Unclear\n3: Slightly below-average clarity\n4: Moderate clarity\n5: Above-average clarity\n6: Clear\n7: Very clear"
            }
        }
    }
}


# New system prompt
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_FORMATIVE,
    suite_description="Formative NoBARS, including requirements,scale 7, system prompt V3, user prompt V3",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_BARTER_NEW2)
formative_suite.save(
    suite_folder, filename="formative_v3_continuous_noBARS_wrequirements_scale7_SPV3_UPV3")

✅ Saved suite to ..\prompts\PromptSuites\BARTER_DEALS\formative_v3_continuous_noBARS_wrequirements_scale7_SPV3_UPV3_suite_aaed425bb6d8.yml


WindowsPath('../prompts/PromptSuites/BARTER_DEALS/formative_v3_continuous_noBARS_wrequirements_scale7_SPV3_UPV3_suite_aaed425bb6d8.yml')

# McGill FeedbackQA

## Init

In [22]:
suite_folder = Path("../prompts/PromptSuites/MCGILL_QA_FEEDBACK")
pm = PromptManager(suite_folder)

PromptManager initialized with folder: ..\prompts\PromptSuites\MCGILL_QA_FEEDBACK


In [23]:
SYSTEM_PROMPT_QA = """You are an expert evaluator of Question and Answer (Q&A) pairs for a public health information platform.
In this ecosystem, individuals ask questions regarding the COVID-19 pandemic, and an informational text passage is provided as the answer. Ground your evaluation in this specific context of health-related information exchange.

You will be given one Q&A Pair consisting of a Question and an Answer. Evaluate it on exactly one dimension: {dim_key_clean}.

Dimension Definition:
{definition}

Scale Definition:
{rubric}

Rate the Q&A Pair on this dimension using the scale above.
Output only the integer score."""

## Holistic

### Holistic naive

In [24]:
EVALUATION_CONFIG_HOLISTIC_NAIVE = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa", "holistic_naive"],
        "dimensions": {
            "overall_answer_quality": {
                "definition": (
                    "Overall Answer Quality evaluates the overall degree to which the answer functions as a good answer to the question in a public health information setting. "
                    "It measures how satisfactory the answer is as a response to the question, taking into account how well it serves the user's information need in context. "
                    "This ranges from an answer that fails to function as a satisfactory answer to an answer that functions very well as a satisfactory answer."
                ),
                "rubric": "1: Bad\n2: Could be Improved\n3: Good\n4: Excellent"
            }
        }
    }
}

# New system prompt
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_HOLISTIC_NAIVE,
    suite_description="Holistic naive v1, scale 4, system prompt V1, user prompt V1",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA)
formative_suite.save(
    suite_folder, filename="holistic_naive_v1_scale4_SPV1_UPV1")

✅ Saved suite to ..\prompts\PromptSuites\MCGILL_QA_FEEDBACK\holistic_naive_v1_scale4_SPV1_UPV1_suite_712743e7a514.yml


WindowsPath('../prompts/PromptSuites/MCGILL_QA_FEEDBACK/holistic_naive_v1_scale4_SPV1_UPV1_suite_712743e7a514.yml')

In [33]:
EVALUATION_CONFIG_HOLISTIC_NAIVE = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa", "holistic_naive"],
        "dimensions": {
            "overall_answer_quality": {
                "definition": (
                    "Overall Answer Quality evaluates the overall degree to which the answer functions as a good answer to the question in a public health information setting. "
                    "It concerns the overall adequacy of the answer as a response to the question. "
                    "This ranges from an answer that fails to function as a satisfactory answer to an answer that functions very well as a satisfactory answer."
                ),
                "rubric": "1: Bad\n2: Could be Improved\n3: Good\n4: Excellent"
            }
        }
    }
}

# New system prompt
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_HOLISTIC_NAIVE,
    suite_description="Holistic naive v2, scale 4, system prompt V1, user prompt V2",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA)
formative_suite.save(
    suite_folder, filename="holistic_naive_v2_scale4_SPV1_UPV2")

✅ Saved suite to ..\prompts\PromptSuites\MCGILL_QA_FEEDBACK\holistic_naive_v2_scale4_SPV1_UPV2_suite_f926cb88b5c8.yml


WindowsPath('../prompts/PromptSuites/MCGILL_QA_FEEDBACK/holistic_naive_v2_scale4_SPV1_UPV2_suite_f926cb88b5c8.yml')

### Holistic informed

In [25]:
EVALUATION_CONFIG_HOLISTIC_INFORMED = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa", "holistic_informed"],
        "dimensions": {
            "overall_answer_quality": {
                "definition": (
                    "Overall Answer Quality evaluates the overall degree to which the answer functions as a good answer to the question in a public health information setting. "
                    "It measures how well the answer addresses the asked question, provides enough information to serve as an answer, matches the specific case described in the question, and stays focused on the requested topic without substantial irrelevant content. "
                    "This ranges from an answer that fails to function as a satisfactory answer because it does not adequately address the question, lacks needed information, does not fit the specific case, or includes substantial off-target content to an answer that directly, sufficiently, specifically, and appropriately answers the question."
                ),
                "rubric": "1: Bad\n2: Could be Improved\n3: Good\n4: Excellent"
            }
        }
    }
}

# New system prompt
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_HOLISTIC_INFORMED,
    suite_description="Holistic informed v1, scale 4, system prompt V1, user prompt V1",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA)
formative_suite.save(
    suite_folder, filename="holistic_informed_v1_scale4_SPV1_UPV1")

✅ Saved suite to ..\prompts\PromptSuites\MCGILL_QA_FEEDBACK\holistic_informed_v1_scale4_SPV1_UPV1_suite_164e78fc2c32.yml


WindowsPath('../prompts/PromptSuites/MCGILL_QA_FEEDBACK/holistic_informed_v1_scale4_SPV1_UPV1_suite_164e78fc2c32.yml')

In [30]:
EVALUATION_CONFIG_HOLISTIC_INFORMED = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa", "holistic_informed"],
        "dimensions": {
            "overall_answer_quality": {
                "definition": "Overall Answer Quality evaluates the overall degree to which the answer functions as a good answer to the question in a public health information setting. It concerns whether the answer matches the issue and constraints stated in the question, provides enough relevant information to serve as a full answer, remains centered on the asked issue without unnecessary extra material, and is expressed clearly enough to be easily understood. This ranges from an answer that functions poorly as an answer because it is misaligned, incomplete, unfocused, or unclear to an answer that functions very well as an answer because it is aligned, sufficiently informative, well focused, and clearly communicated.",
                "rubric": "1: Bad\n2: Could be Improved\n3: Good\n4: Excellent"
            }
        }
    }
}

# New system prompt
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_HOLISTIC_INFORMED,
    suite_description="Holistic informed v2, scale 4, system prompt V1, user prompt V2, rubric v2",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA)
formative_suite.save(
    suite_folder, filename="holistic_informed_v2_scale4_SPV1_UPV2_RUBV2")

✅ Saved suite to ..\prompts\PromptSuites\MCGILL_QA_FEEDBACK\holistic_informed_v2_scale4_SPV1_UPV2_RUBV2_suite_f1ea5fbc4957.yml


WindowsPath('../prompts/PromptSuites/MCGILL_QA_FEEDBACK/holistic_informed_v2_scale4_SPV1_UPV2_RUBV2_suite_f1ea5fbc4957.yml')

## Formative

In [32]:
EVALUATION_CONFIG_FORMATIVE = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa", "formative"],
        "dimensions": {
            "question_responsiveness": {
                "definition": (
                    "Question Responsiveness evaluates whether the answer addresses the question that was asked. "
                    "It measures the degree to which the answer responds to the requested issue, rather than discussing a different issue, only a loosely related issue, or only part of the asked issue. "
                    "This ranges from an answer that fails to address the asked question or mainly addresses a different issue to an answer that directly addresses the asked question."
                ),
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            },
            "information_sufficiency": {
                "definition": (
                    "Information Sufficiency evaluates whether the answer provides enough information to serve as a satisfactory answer. "
                    "It measures the degree to which the answer contains the needed content, details, or conditions, rather than being too limited, underspecified, or incomplete to satisfy the informational demand of the question. "
                    "This ranges from an answer that lacks necessary information and leaves major informational gaps to an answer that provides enough information for a full answer."
                ),
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            },
            "case_specificity": {
                "definition": (
                    "Case Specificity evaluates whether the answer matches the specific case described in the question. "
                    "It measures the degree to which the answer is tailored to the exact situation, group, condition, or constraint expressed in the question, rather than remaining generic or addressing only a broader related case. "
                    "This ranges from an answer that does not match the specific case being asked about to an answer that is well matched to the exact case described in the question."
                ),
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            },
            "topical_focus": {
                "definition": (
                    "Topical Focus evaluates whether the answer stays concentrated on the information needed for the question. "
                    "It measures the degree to which the answer remains on the requested topic, rather than including substantial irrelevant, distracting, or off-target content that weakens the answer as a response to the question. "
                    "This ranges from an answer dominated by irrelevant or off-target content to an answer that stays tightly focused on the requested topic."
                ),
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            }
        }
    }
}

# New system prompt
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_FORMATIVE,
    suite_description="Formative v2, scale 4, system prompt V1, user prompt V1, rubric V1",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA)
formative_suite.save(
    suite_folder, filename="formative_v2_scale4_SPV1_UPV1_RUBV1")

✅ Saved suite to ..\prompts\PromptSuites\MCGILL_QA_FEEDBACK\formative_v2_scale4_SPV1_UPV1_RUBV1_suite_be88483950b8.yml


WindowsPath('../prompts/PromptSuites/MCGILL_QA_FEEDBACK/formative_v2_scale4_SPV1_UPV1_RUBV1_suite_be88483950b8.yml')

In [34]:
EVALUATION_CONFIG_FORMATIVE = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa", "formative"],
        "dimensions": {
            "topical_alignment": {
                "definition": "Topical Alignment evaluates the degree to which the answer addresses the issue and stated constraints of the question. It concerns whether the answer matches the subject matter, target case, and requested context expressed in the question. This ranges from an answer that addresses a different issue or misses the requested context to an answer that directly matches the issue and constraints of the question.",
                "rubric": "1: Bad\n2: Could be Improved\n3: Good\n4: Excellent"
            },
            "information_coverage": {
                "definition": "Information Coverage evaluates the degree to which the answer provides enough relevant information to serve as a full answer. It concerns the completeness of the relevant content provided in relation to the question. This ranges from an answer that leaves major informational gaps to an answer that covers the relevant aspects needed for a full answer.",
                "rubric": "1: Bad\n2: Could be Improved\n3: Good\n4: Excellent"
            },
            "topical_concentration": {
                "definition": "Topical Concentration evaluates the degree to which the answer remains centered on the asked issue throughout the passage. It concerns how consistently the content stays on the requested topic without drifting into extra material. This ranges from an answer with substantial off-topic or unnecessary content to an answer that stays tightly centered on the asked issue.",
                "rubric": "1: Bad\n2: Could be Improved\n3: Good\n4: Excellent"
            },
            "communicative_clarity": {
                "definition": "Communicative Clarity evaluates the degree to which the answer is expressed in a clear, well-organized, and easy-to-follow way. It concerns how readily a reader can understand the information from the wording and structure of the passage. This ranges from an answer that is difficult to follow or poorly expressed to an answer that is clearly written and easy to understand.",
                "rubric": "1: Bad\n2: Could be Improved\n3: Good\n4: Excellent"
            }
        }
    }
}
# New system prompt
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_FORMATIVE,
    suite_description="Formative v3, scale 4, system prompt V1, user prompt V1, rubric V2",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA)
formative_suite.save(
    suite_folder, filename="formative_v3_scale4_SPV1_UPV1_RUBV2")

✅ Saved suite to ..\prompts\PromptSuites\MCGILL_QA_FEEDBACK\formative_v3_scale4_SPV1_UPV1_RUBV2_suite_83fb9645af53.yml


WindowsPath('../prompts/PromptSuites/MCGILL_QA_FEEDBACK/formative_v3_scale4_SPV1_UPV1_RUBV2_suite_83fb9645af53.yml')